# einops-rearrange-flatten — worked example 1: Flatten sequence and embedding axes into a single feature vector per batch item

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-rearrange-flatten`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`einops.rearrange` with a parenthesized group on the right side merges those axes into one. The pattern `'b t d -> b (t d)'` collapses the sequence length `t` and embedding dimension `d` into a single flat feature axis, while keeping the batch axis `b` intact. This is the same operation as `x.flatten(start_dim=1)` but expressed with explicit axis names.

## Worked solution

Input shape: `(B=2, T=4, D=8)`. We want `(B, T*D) = (2, 32)`.

**Pattern:** `'b t d -> b (t d)'`.

Einops reads the right side: `b` survives as-is, `(t d)` means merge `t` and `d` into a single axis of size `T*D = 32`. The merge is row-major — element `[b, t, d]` maps to flat position `t*D + d`.

**Why use einops?** The pattern string documents what the axes mean. When you come back to the code later, `'b t d -> b (t d)'` communicates that you're flattening the sequence-embedding space, not just any two trailing dims.

In [ ]:
import torch as t
from einops import rearrange

t.manual_seed(17)
B, T, D = 3, 5, 16
x = t.randn(B, T, D)

def flatten_seq_embed(x):
    return rearrange(x, 'b t d -> b (t d)')

flat = flatten_seq_embed(x)
print('Input shape:', x.shape)   # (3, 5, 16)
print('Output shape:', flat.shape)  # (3, 80)
assert flat.shape == (B, T * D)

# Verify element ordering matches flatten(start_dim=1)
assert t.allclose(flat, x.flatten(start_dim=1))
print('Matches x.flatten(start_dim=1):', True)